In [ ]:
pip install --upgrade git+https://@github.com/blsq/blsq_dqapp.git

In [ ]:
import gc
import os

import numpy as np
import pandas as pd
from sqlalchemy import create_engine

import blsq_dqapp as dqapp

In [ ]:
YEAR = 2026
COMPILE_OUTPUT = False

In [ ]:
# project_name = "dqapp_blsq_pnlp"
# project_path_root = "s3://hexa-cod-msp/data/"
# project_path_raw = project_path_root + "snis-dhis/"
# project_path_processed = project_path_raw + "drc_malaria_drug_logistics/"
# files_main_name_output = project_name + "-latest"

project_name = "dqapp_blsq_pnlp"
project_path_root = "/home/hexa/workspace/pnlp-tdb-pipeline/"
project_path_interim = project_path_root + "data/interim/"
project_path_processed = project_path_interim + "availability/"

In [ ]:
source_file_name = "pnlp_indicators_availability"

# if YEAR in [2018, 2019]:
#     year_suffix = "2018-2019"
# else:
#     year_suffix = str(YEAR) 

In [ ]:
meta_path = f"{project_path_root}data/metadata/"
date_suffix = "_2021-03-23.csv"

# OU_Tree extracted through API from DRC DHIS2
org_unit_tree = pd.read_csv(meta_path + "DRC_organisationUnitsStructure" + date_suffix)

# DS DE/OU list extracted through API from DRC DHIS2
datelements_metadata = pd.read_csv(meta_path + "DRC_dataElementsStructure" + date_suffix)
datasets_datelements_orgunit = pd.read_csv(meta_path + "DRC_dataSetsStructure_organisationUnits" + date_suffix)

In [ ]:
ind_metadata = pd.read_csv(
    project_path_root + "data/metadata/pnlp_indicators_avail_dict.csv",
    usecols=["ID_N", "DE_UID", "TYPE"],
)
de_filter = ind_metadata.DE_UID.unique().tolist()
datelements_metadata_filtered = datelements_metadata.query("DE_UID in @de_filter")

In [ ]:
# if YEAR > 2021:
#     input_cols = ["ou_uid", "period", "value", "dx_uid", "coc_uid"]
    
#     extract_paths = [
#         ('/home/jovyan/s3-hexa-cod-msp/data/snis-dhis/pnlp-indicators/routine/'
#          f'{year_suffix}/Q{x}/extract.csv') for x in range(1, 5)
#     ]
    
#     extract_paths = [x for x in extract_paths if os.path.exists(x)]

#     data_raw = pd.concat([pd.read_csv(p) for p in extract_paths])
    
#     data_raw.rename(columns={'dx_uid': 'de_uid'}, 
#                inplace=True)
#     data_raw.columns = data_raw.columns.str.upper()
    
# else:
#     input_cols = ["OU_UID", "PERIOD", "VALUE", "DE_UID", "COC_UID"]

#     data_raw = pd.concat(
#         [
#             pd.read_csv(
#                 project_path_raw
#                 + "pnlp-indicators/pnlp_indicators_"
#                 + year_suffix
#                 + ".csv",
#                 usecols=input_cols,
#             )
#         ],
#         ignore_index=True,
#     )

input_cols = ["OU_UID", "PERIOD", "VALUE", "DE_UID", "COC_UID"]

extract_paths = [
    f'{project_path_interim}PNLP_{x}_{str(YEAR)}.feather' for x in ['indicators', 
                                                                    'logistics']
]

data_raw = pd.concat([pd.read_feather(p) for p in extract_paths])
    

data_raw = data_raw.query("DE_UID in @de_filter")
# filter_m=data_raw.PERIOD.astype(str).apply(lambda x : 'Q' in x)
# data_raw=data_raw[~filter_m]
data_raw.PERIOD = data_raw.PERIOD.astype(int)

In [ ]:
data_indicator = pd.concat(
    [
        pd.read_feather((f'{project_path_interim}all_cause_mortality_'
                    f'{str(YEAR)}.feather'))
    ],
    ignore_index=True,
)
data_indicator = data_indicator.rename(columns={"IND_UID": "DE_UID"})[
    ["OU_UID", "PERIOD", "VALUE", "DE_UID"]
]
data_indicator.PERIOD = data_indicator.PERIOD.astype(int)
data_indicator.VALUE = data_indicator.VALUE.astype(float)
data_indicator["COC_UID"] = "IND"

In [ ]:
data_raw_filtered = dqapp.formatting_tools.data_values_trimmering(data_raw)
period_table = data_raw_filtered[["PERIOD"]].drop_duplicates()
data_raw_filtered.VALUE = data_raw_filtered.VALUE.astype(str).astype(float)

In [ ]:
data_raw_filtered = pd.concat([data_raw_filtered, data_indicator], ignore_index=True)

In [ ]:
availability_tree_table = dqapp.availability.availability_tree_table_maker(
    datasets_datelements_orgunit, datelements_metadata_filtered, period_table
)

In [ ]:
ind12_num_de_list = ["wfmDVt6RVm2", "nFjuJLLeejV"]

ind_12_den_expected = availability_tree_table.query("DE_UID in @ind12_num_de_list")[
    ["OU_UID", "PERIOD"]
].drop_duplicates()
ind_12_den_expected["DE_UID"] = "fvlFcxuGRng"
ind_12_den_expected["COC_UID"] = "IND"

In [ ]:
availability_tree_table = pd.concat(
    [availability_tree_table, ind_12_den_expected], ignore_index=True
)

In [ ]:
cmr_gff_ind_dqapp = dqapp.quality_container.quality_auction_container(
    data_raw_filtered,
    org_unit_tree,
    datelements_metadata_filtered,
    project_path_processed=project_path_processed,
    files_main_name=source_file_name,
    de_uid_vars=["DE_UID", "COC_UID"],
)

In [ ]:
cmr_gff_ind_dqapp.availability_generation(custom_tree_input=availability_tree_table)
cmr_gff_ind_dqapp.processed_df.head()

In [ ]:
de_availability = (
    cmr_gff_ind_dqapp.processed_df.groupby(["DE_UID", "PERIOD", "OU_UID"])
    .DE_AVAILABILITY_BOOL.max()
    .reset_index()
)

In [ ]:
ind_availability = ind_metadata.merge(de_availability, on="DE_UID")

In [ ]:
ind_availability_atleast_attype = (
    ind_availability.groupby(["ID_N", "TYPE", "PERIOD", "OU_UID"])
    .DE_AVAILABILITY_BOOL.max()
    .reset_index()
)
ind_availability_atleast_attype = (
    ind_availability_atleast_attype.groupby(["ID_N", "PERIOD", "OU_UID"])
    .DE_AVAILABILITY_BOOL.min()
    .reset_index()
)

In [ ]:
ind_availability_atleast_attype = ind_availability_atleast_attype.rename(
    columns={"DE_AVAILABILITY_BOOL": "AVB_LEAST_TYPE"}
)

In [ ]:
ind_availability_absolute = (
    ind_availability.groupby(["ID_N", "PERIOD", "OU_UID"])
    .DE_AVAILABILITY_BOOL.min()
    .reset_index()
)

In [ ]:
ind_availability_absolute = ind_availability_absolute.rename(
    columns={"DE_AVAILABILITY_BOOL": "AVB_ABSOLUTE"}
)

In [ ]:
ind_availability_global = ind_availability_absolute.merge(
    ind_availability_atleast_attype, on=["ID_N", "PERIOD", "OU_UID"]
)

In [ ]:
ind_metadata_dict = pd.read_csv(
    project_path_root + "data/metadata/pnlp_indicators_avail_dict.csv",
    usecols=["ID_N", "ID_NAME"],
)
ind_metadata_dict.ID_N = ind_metadata_dict.ID_N.astype(str)

In [ ]:
ind_availability_global_f = ind_availability_global.merge(ind_metadata_dict, on="ID_N")[
    ["ID_N", "ID_NAME", "PERIOD", "OU_UID", "AVB_LEAST_TYPE", "AVB_ABSOLUTE"]
]

In [ ]:
ind_availability_global_f["TOTAL FOSA"] = 1

In [ ]:
ind_availability_global_f.to_feather(
    project_path_processed
    + "indicators_availability_level_5_month_newformat_"
    + str(YEAR)
    + ".feather"
)

In [ ]:
ou_zs_level = org_unit_tree[
    ["OU_UID", "LEVEL_3_UID", "LEVEL_3_NAME", "LEVEL_2_UID", "LEVEL_2_NAME"]
].drop_duplicates()
ind_availability_global_zs = (
    ind_availability_global_f.merge(ou_zs_level, on="OU_UID")
    .groupby(
        [
            "ID_N",
            "ID_NAME",
            "PERIOD",
            "LEVEL_2_UID",
            "LEVEL_2_NAME",
            "LEVEL_3_UID",
            "LEVEL_3_NAME",
        ]
    )
    .sum()
    .reset_index()
)  # mean().reset_index()

In [ ]:
ind_availability_global_zs.to_feather(
    project_path_processed
    + "indicators_availability_level_3_month_newformat_"
    + str(YEAR)
    + ".feather",
)

In [ ]:
ou_as_level = org_unit_tree[
    [
        "OU_UID",
        "LEVEL_4_UID",
        "LEVEL_4_NAME",
        "LEVEL_3_UID",
        "LEVEL_3_NAME",
        "LEVEL_2_UID",
        "LEVEL_2_NAME",
    ]
].drop_duplicates()
ind_availability_global_as = (
    ind_availability_global_f.merge(ou_as_level, on="OU_UID")
    .drop('OU_UID', axis=1)
    .groupby(
        [
            "ID_N",
            "ID_NAME",
            "PERIOD",
            "LEVEL_2_UID",
            "LEVEL_2_NAME",
            "LEVEL_3_UID",
            "LEVEL_3_NAME",
            "LEVEL_4_UID",
            "LEVEL_4_NAME",
        ]
    )
    .sum()
    .reset_index()
)  # mean().reset_index()

In [ ]:
ind_availability_global_as.to_feather(
    project_path_processed
    + "indicators_availability_level_4_month_newformat_"
    + str(YEAR)
    + ".feather",
)

### CONCILIATION OF ALL PIECES (deprecated)

In [ ]:
# if COMPILE_OUTPUT:
#     ind_availability_global_f = pd.concat(
#         [
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_5_month_newformat_2018-2019.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_5_month_newformat_2020.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_5_month_newformat_2021.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_5_month_newformat_2022.csv"
#             )
#         ],
#         ignore_index=True,
#     )
#     ind_availability_global_f.to_csv(
#         project_path_processed
#         + "availability/indicators_availability_level_5_month_newformat.csv",
#         index=False,
#     )

In [ ]:
# if COMPILE_OUTPUT:
#     ind_availability_global_zs = pd.concat(
#         [
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_3_month_newformat_2018-2019.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_3_month_newformat_2020.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_3_month_newformat_2021.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_3_month_newformat_2022.csv"
#             )
#         ],
#         ignore_index=True,
#     )
#     ind_availability_global_zs.to_csv(
#         project_path_processed
#         + "availability/indicators_availability_level_3_month_newformat.csv",
#         index=False,
#     )

In [ ]:
# if COMPILE_OUTPUT:
#     ind_availability_global_as = pd.concat(
#         [
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_4_month_newformat_2018-2019.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_4_month_newformat_2020.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_4_month_newformat_2021.csv"
#             ),
#             pd.read_csv(
#                 project_path_processed
#                 + "availability/indicators_availability_level_4_month_newformat_2022.csv"
#             )
#         ],
#         ignore_index=True,
#     )
#     ind_availability_global_as.to_csv(
#         project_path_processed
#         + "availability/indicators_availability_level_4_month_newformat.csv",
#         index=False,
#     )